# 14-day cumulative number of COVID-19 cases per 100 000 for districts (Landkreise) in Germany

At the end of the page, we provide a detailed description of how the numbers are calculated.

### Compute data

In [1]:
import datetime as dt
import pandas as pd
pd.set_option("max_rows", None)
from oscovida import get_incidence_rates_germany

## Table for all districts (=Landkreise)

In [2]:
cases_incidence, deaths_incidence = get_incidence_rates_germany()

Please be patient - downloading data from https://www.arcgis.com/sharing/rest/content/items/f10774f1c63e40168479a1feb6c7ca74/data ...
Completed downloading 221014 rows in 5.1 seconds.


In [3]:
cases_incidence

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
LK Ahrweiler,17,129727,13.1
LK Aichach-Friedberg,21,133596,15.7
LK Alb-Donau-Kreis,64,196047,32.6
LK Altenburger Land,7,90118,7.8
LK Altenkirchen,15,128705,11.7
LK Altmarkkreis Salzwedel,4,83765,4.8
LK Altötting,18,111210,16.2
LK Alzey-Worms,19,129244,14.7
LK Amberg-Sulzbach,19,103109,18.4


## Table sorted by 14-day-incidence

In [4]:
cases_incidence.sort_values(by="14-day-incidence-rate", ascending=False)

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Würzburg,157,127880,122.8
SK Hamm,166,179111,92.7
LK Cloppenburg,156,169348,92.1
SK München,1321,1471508,89.8
SK Kaufbeuren,37,43893,84.3
SK Berlin Friedrichshain-Kreuzberg,228,289762,78.7
SK Remscheid,85,110994,76.6
LK Garmisch-Partenkirchen,65,88467,73.5
LK Neu-Ulm,125,174200,71.8


## Tutorial: Detailed calculation for one district

In [5]:
from oscovida import fetch_data_germany, germany_get_population
import datetime

In [6]:
period = 14 # Set the period we compute the incidence rate over

In [7]:
germany = fetch_data_germany() # Downloads population data for Germany
germany = germany.rename(
    columns={"AnzahlFall": "cases", "AnzahlTodesfall": "deaths"}
) # Rename the columns to English
germany.tail() # Print example of the table format and data

,FID,IdBundesland,Bundesland,Landkreis,Altersgruppe,Geschlecht,cases,deaths,Meldedatum,IdLandkreis,Datenstand,NeuerFall,NeuerTodesfall,Refdatum,NeuGenesen,AnzahlGenesen,IstErkrankungsbeginn,Altersgruppe2
date,,,,,,,,,,,,,,,,,,
2020-06-09,37661579,16,Thüringen,LK Altenburger Land,A80+,M,1,0,2020/06/09 00:00:00,16077,"23.09.2020, 00:00 Uhr",0,-9,2020/05/19 00:00:00,0,1,1,Nicht übermittelt
2020-05-06,37661580,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/06 00:00:00,16077,"23.09.2020, 00:00 Uhr",0,-9,2020/05/04 00:00:00,0,1,1,Nicht übermittelt
2020-05-11,37661581,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/05/11 00:00:00,16077,"23.09.2020, 00:00 Uhr",0,-9,2020/05/01 00:00:00,0,1,1,Nicht übermittelt
2020-05-28,37661582,16,Thüringen,LK Altenburger Land,A80+,W,2,0,2020/05/28 00:00:00,16077,"23.09.2020, 00:00 Uhr",0,-9,2020/05/28 00:00:00,0,2,0,Nicht übermittelt
2020-06-09,37661583,16,Thüringen,LK Altenburger Land,A80+,W,1,0,2020/06/09 00:00:00,16077,"23.09.2020, 00:00 Uhr",0,-9,2020/06/09 00:00:00,0,1,0,Nicht übermittelt


In [8]:
yesterday = datetime.datetime.now() - datetime.timedelta(days=1)
fortnight_ago = yesterday - datetime.timedelta(days=period)
periods = (fortnight_ago < germany.index) & (germany.index < yesterday) # Create a period mask
germany = germany.iloc[periods]

In [9]:
cases_germany = germany[["Landkreis", "cases"]]
cases_germany.tail() # The data for germany has the date as the index, and the location data on each column

,Landkreis,cases
date,,
2020-09-12,LK Altenburger Land,1
2020-09-12,LK Altenburger Land,2
2020-09-15,LK Altenburger Land,1
2020-09-22,LK Altenburger Land,1
2020-09-12,LK Altenburger Land,1


In [10]:
cases_sum = (
    cases_germany.groupby("Landkreis")
    .sum()
    .rename(columns={"cases": f"{period}-day-sum"})
) # Here we sum the daily data to get total data
cases_sum.tail()

,14-day-sum
Landkreis,
SK Worms,42
SK Wuppertal,138
SK Würzburg,157
SK Zweibrücken,8
StadtRegion Aachen,184


In [11]:
population = (
    germany_get_population()
    .population
    .to_frame()
) # This gives us population per county, with county as the index
population.tail()

,population
county,
SK Berlin Spandau,243977
SK Berlin Steglitz-Zehlendorf,308697
SK Berlin Mitte,384172
SK Berlin Friedrichshain-Kreuzberg,289762
SK Berlin Tempelhof-Schöneberg,351644


In [12]:
cases_incidence = cases_sum.join(population)
cases_incidence[f"{period}-day-incidence-rate"] = (
    cases_incidence[f"{period}-day-sum"] / cases_incidence["population"] * 100_000
) # By convention the incedence rate is the total cases in the past x days / population * 100_000

In [13]:
cases_incidence.tail()

,14-day-sum,population,14-day-incidence-rate
Landkreis,,,
SK Worms,42,83330,50.402016
SK Wuppertal,138,354382,38.941030
SK Würzburg,157,127880,122.771348
SK Zweibrücken,8,34209,23.385659
StadtRegion Aachen,184,555465,33.125399


In [14]:
cases_incidence.loc['SK Hamburg']

14-day-sum               6.390000e+02
population               1.841179e+06
14-day-incidence-rate    3.470602e+01
Name: SK Hamburg, dtype: float64